# How to use
1. Create huggingface account
2. create a read token and a read_write token
3. add it to secrets (key on left) as HF_TOKEN and HF_TOKEN_ALL
4. set `USE_TESSERA`, `TS_LENGTH`, `TS_INTERVAL`, `TRAIN_VAL_LABEL_SEL` and the repo names in the first code cell, run the setup cells, then uncomment the step you need
5. data will be saved

All logic lives in the repo's `firepred/hub/` package (`publish_windows`, `publish_tessera`, `download`, `zips`); this notebook only supplies secrets and paths.

**Hugging Face datasets** (defaults below; change any of them in the first code cell):
| constant | repo | contents |
|---|---|---|
| `SRC_REPO` | `SamuelWu318/ts-satfire` | raw GeoTIFFs + ROI CSVs |
| `WINDOW_REPO` | `SamuelWu318/ts-satfire-processed-6-1` | windowed NPZ per fire + `data.zip`; use one repo per `TS_LENGTH` / `TS_INTERVAL` setting |
| `TESSERA_REPO` | `SamuelWu318/ts-tesserafire` | per-fire TESSERA embeddings + `tessera.zip` |

**Storage:** every download, cache, and temporary file stays on this Colab VM under `/content` (`WORK_DIR`). Colab's disk is wiped when the session ends, so finished data is uploaded to Hugging Face.

**TESSERA mode** (`USE_TESSERA = True`): PULL ZIP adds `tessera.zip` (embeddings + drop list) next to `data.zip` in its own `DATA_ROOT`, then removes the window archives of fires with no prior-year TESSERA (all 2017 fires). Window data is stored once, in `WINDOW_REPO`'s `data.zip`.

In [1]:
# Pin every cache under /content before anything imports huggingface_hub or geotessera.
import os

USE_TESSERA = False  # True: training data gets per-fire TESSERA embeddings and drops 2017 fires

# Window settings (defaults in firepred/config/data.py). A fire needs TS_LENGTH + 1 days.
# The TS-SatFire paper's prediction benchmark uses 6 input days. Each setting has its own WINDOW_REPO.
TS_LENGTH = 6    # input days per window
TS_INTERVAL = 1   # days between window starts

# Train/val burned-area labels (TRAIN_VAL_LABEL_SEL in firepred/config/data.py).
#   1: accumulated active-fire (AF) detections only, no NIFC perimeters, as the official code
#      builds them. Used here.
#   0: AF detections + NIFC perimeters, as the paper's text describes.
# Test fires keep each fire's rule from the 2021 fire list (as in the paper and official code).
# Processing skips fires already in WINDOW_REPO, so delete train/ and val/ archives built with
# the other rule (and the old data.zip) before rebuilding; each archive records its label_sel.
TRAIN_VAL_LABEL_SEL = 1
assert TRAIN_VAL_LABEL_SEL in (0, 1)

# Hugging Face dataset repos ("owner/name"). Change any of them to use your own.
SRC_REPO = "SamuelWu318/ts-satfire"                    # raw GeoTIFFs + ROI CSVs (read)
TESSERA_REPO = "SamuelWu318/ts-tesserafire"            # per-fire TESSERA + tessera.zip
WINDOW_REPO = "SamuelWu318/ts-satfire-processed-6-1"  # windows for TS_LENGTH / TS_INTERVAL above

WORK_DIR = "/content"
DATA_ROOT = f"{WORK_DIR}/data_ts{TS_LENGTH}_it{TS_INTERVAL}" + ("_tessera" if USE_TESSERA else "")
ROI_DIR = f"{WORK_DIR}/roi"
CACHE_DIR = f"{WORK_DIR}/cache"

# Two fires with prior-year TESSERA for the timing test: a 2018 train fire and a 2021 test fire.
TEST_FIRES = ["21890003", "US_2021_MT4714310953420211004"]

os.environ["HF_HOME"] = f"{CACHE_DIR}/huggingface"
os.environ["XDG_CACHE_HOME"] = CACHE_DIR
os.environ["TMPDIR"] = f"{CACHE_DIR}/tmp"
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

In [2]:
REPO_DIR = f"{WORK_DIR}/fire-research-and-innovation"
BRANCH = "explain"  # branch with the firepred package; "main" once it is merged

!git clone -b {BRANCH} https://github.com/SamuelWu318/fire-research-and-innovation.git {REPO_DIR} 2>/dev/null || (git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH})
!git -C {REPO_DIR} log --oneline -1
%cd {REPO_DIR}
!pip install -q -U huggingface_hub imagecodecs rasterio pyyaml pandas "geotessera>=0.11"

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 363 bytes | 51.00 KiB/s, done.
From https://github.com/SamuelWu318/fire-research-and-innovation
   e2cc52d..9d7a772  main       -> origin/main
Already on 'embeddings'
Your branch is up to date with 'origin/embeddings'.
From https://github.com/SamuelWu318/fire-research-and-innovation
 * branch            embeddings -> FETCH_HEAD
Already up to date.
dbf760e (HEAD -> embeddings, origin/embeddings) added f1 evaluations to wandb
/content/fire-research-and-innovation


In [3]:
from google.colab import userdata

from firepred.hub.download import download_processed_data
from firepred.hub.publish_tessera import process_tessera, upload_tessera_zip
from firepred.hub.publish_windows import process_data
from firepred.hub.splits import obtain_ids, tessera_excluded_npz
from firepred.hub.zips import pull_tessera_data, pull_zip_data_to, upload_data_zip

TOKEN = userdata.get("HF_TOKEN")          # read
TOKEN_ALL = userdata.get("HF_TOKEN_ALL")  # read + write

print("SRC_REPO:          ", SRC_REPO)
print("WINDOW_REPO:       ", WINDOW_REPO, f"({TS_LENGTH} days every {TS_INTERVAL})")
print("TESSERA_REPO:      ", TESSERA_REPO)
print("DATA_ROOT:         ", DATA_ROOT)

SRC_REPO:           SamuelWu318/ts-satfire
WINDOW_REPO:        SamuelWu318/ts-satfire-processed-6-1 (6 days every 1)
TESSERA_REPO:       SamuelWu318/ts-tesserafire
PROCESSED_REPO:     SamuelWu318/ts-satfire-processed
LEGACY_WINDOW_REPO: SamuelWu318/ts-satfire-processed-window
DATA_ROOT:          /content/data_ts6_it1


# DATA GRAB
Streams raw GeoTIFFs from `SRC_REPO` into one windowed NPZ per fire in `WINDOW_REPO` (`train/`, `val/`, `test/`), using `TS_LENGTH` input days every `TS_INTERVAL` days. Train/val labels follow `TRAIN_VAL_LABEL_SEL` (1, used here: active-fire detections only, as the official code; 0: detections + NIFC perimeters, as the paper's text). Each archive records the rule that built it; delete archives built with the other rule before rebuilding, since existing fires are skipped. The repo records its settings in `window_config.json` and refuses windows with other settings. Only run if the processed dataset is empty; fires already in `WINDOW_REPO` are skipped.

In [4]:
# debugging split
from firepred.hub.publish_windows import DatasetStreamer
# DatasetStreamer(SRC_REPO, WINDOW_REPO, TOKEN_ALL, work_dir=WORK_DIR).run_split(
#     ['20568194', '20701026', '20562846', '20700973', '24462610'], "debug")

splits = ("test", "train", "val")
print(f"Train/val label rule {TRAIN_VAL_LABEL_SEL} -> {WINDOW_REPO}")
process_data(TOKEN_ALL, src_repo=SRC_REPO, window_repo=WINDOW_REPO,
             length=TS_LENGTH, interval=TS_INTERVAL,
             work_dir=WORK_DIR, roi_dir=ROI_DIR, splits=splits,
             batch_size=3, train_val_label_sel=TRAIN_VAL_LABEL_SEL)

Windows: 6 input days every 1 days -> SamuelWu318/ts-satfire-processed-6-1


window_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

/content/roi/roi/fire_all.csv
/content/roi/roi/fire_all_test.csv
/content/roi/roi/us_fire_2017_out_new.csv
/content/roi/roi/us_fire_2018_out_new.csv
/content/roi/roi/us_fire_2019_out_new.csv
/content/roi/roi/us_fire_2020_out_new.csv
/content/roi/roi/us_fire_2021_out_new.csv
/content/roi/roi/fire_all.csv
/content/roi/roi/fire_all_test.csv
/content/roi/roi/us_fire_2017_out_new.csv
/content/roi/roi/us_fire_2018_out_new.csv
/content/roi/roi/us_fire_2019_out_new.csv
/content/roi/roi/us_fire_2020_out_new.csv
/content/roi/roi/us_fire_2021_out_new.csv
Train: 125
Val: 13
Test: 24
test: rebuilding 4 processed fires whose label_sel changed: ['US_2021_ID4558511544420210705', 'US_2021_WA4828511853120210713', 'US_2021_WA4856812048820210708', 'US_2021_WA4877811903420210803']
Found 24 valid locations that match.
Skipping US_2021_MT4714310953420211004: already processed
[TEST | 2/24] Processing: US_2021_ID4558511544420210705


ts-satfire/US_2021_ID4558511544420210705(…): reconstructing file:   0%|          |  0.00B / 27.4MB            

ts-satfire/US_2021_ID4558511544420210705(…): downloading bytes:           |  0.00B            

...

ts-satfire/US_2021_WA4828511853120210713(…): reconstructing file:   0%|          |  0.00B / 7.36MB            

ts-satfire/US_2021_WA4828511853120210713(…): downloading bytes:           |  0.00B            

ts-satfire/US_2021_WA4828511853120210713(…): reconstructing file:   0%|          |  0.00B / 7.32MB            

ts-satfire/US_2021_WA4828511853120210713(…): downloading bytes:           |  0.00B            

KeyboardInterrupt: 

# TESSERA
For each fire, reads the crop georeference from its first and last VIIRS_Day GeoTIFF in `SRC_REPO`, streams the prior-year TESSERA embeddings over that crop into RAM, averages them onto the 256x256 grid, and uploads `tessera/p<id>.npz` (embedding + coverage + metadata) to `TESSERA_REPO` after each fire. 2017 fires have no prior-year TESSERA and are skipped.

1. **Timing test** on `TEST_FIRES`; check the `built in N min` lines and the coverage printout.
2. **Full run**; fires already uploaded are skipped, so rerun the cell after a disconnect to resume.
3. **Bundle** `tessera/` plus the drop list into `tessera.zip` in `TESSERA_REPO`; it warns about any kept fire still missing.

In [ ]:
# 1) timing test
# test_skipped = process_tessera(TOKEN_ALL, src_repo=SRC_REPO, tessera_repo=TESSERA_REPO, work_dir=WORK_DIR, roi_dir=ROI_DIR, only_fires=TEST_FIRES)

In [ ]:
# 2) all fires (resumable)
# skipped = process_tessera(TOKEN_ALL, src_repo=SRC_REPO, tessera_repo=TESSERA_REPO, work_dir=WORK_DIR, roi_dir=ROI_DIR)

In [ ]:
# 3) tessera.zip
# missing = upload_tessera_zip(TOKEN_ALL, src_repo=SRC_REPO, tessera_repo=TESSERA_REPO, work_dir=WORK_DIR, roi_dir=ROI_DIR)

# DATA LOADING
Rebuilds `data.zip` in `WINDOW_REPO` from the windowed NPZs. Only needed after DATA GRAB changed them.

In [ ]:
# zip_root = f"{WORK_DIR}/zip_ts{TS_LENGTH}_it{TS_INTERVAL}"
# download_processed_data(TOKEN_ALL, src_repo=SRC_REPO, window_repo=WINDOW_REPO, base_output_dir=zip_root, roi_dir=ROI_DIR)
upload_data_zip(TOKEN_ALL, data_root=zip_root, zip_path=f"{zip_root}.zip", repo_id=WINDOW_REPO)

# PULL ZIP
For training in any Colab: pulls `data.zip` from `WINDOW_REPO` into `DATA_ROOT`. In TESSERA mode it also unpacks `tessera.zip` from `TESSERA_REPO` into `DATA_ROOT/tessera` and removes the 2017 fires' window archives.

In [ ]:
# pull_zip_data_to(DATA_ROOT, token=TOKEN, repo_id=WINDOW_REPO, download_dir=DATA_ROOT)
# if USE_TESSERA:
#     removed = pull_tessera_data(DATA_ROOT, token=TOKEN, tessera_repo=TESSERA_REPO)